# M03-01 — Enriquecimiento

[← Anterior](01-teoria.ipynb) · [Siguiente →](03-lab-reglas-negocio.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Añadir `gmv_line` y `order_month` al cruce líneas ⋈ pedidos y detectar GMV negativo (descuento sucio).

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M03-01-enriquecimiento.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras. No es adorno: es la traza de tu razonamiento.
2. **Código** — lo pegas o lo escribes, lo **ejecutas** (`Shift+Enter`), **miras** la salida y, si no cuadra, lo **mejoras**.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Arranque y staging

**1. Crea una celda Markdown** en *tu* notebook. Explica con tus palabras (puedes partir de esto):

> Leo Parquet de M02-03. El schema ya viaja; no re-inferimos.

**2. Crea una celda de código** debajo y escribe:

```python
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


spark = get_spark("novashop-m03")
orders = spark.read.parquet(str(STAGING / "orders_clean"))
items = spark.read.parquet(str(STAGING / "order_items_clean"))
print(orders.count(), items.count())
```

**3. Ejecuta** esa celda (`Shift+Enter`). Espera a que deje de verse `[*]`.

**4. Comprueba.** `788 2010`.

**Por qué este paso.** Si falla el path, no has escrito staging. Vuelve a M02-03.


### Paso 2 — Inner para tener fecha

**1. Crea una celda Markdown** en *tu* notebook. Explica con tus palabras (puedes partir de esto):

> order_month vive en la cabecera. Inner: las líneas de los 12 pedidos sin cliente no entran.

**2. Crea una celda de código** debajo y escribe:

```python
from pyspark.sql.functions import col

lines = items.join(orders, "order_id", "inner")
print(lines.count())
```

**3. Ejecuta** esa celda (`Shift+Enter`). Espera a que deje de verse `[*]`.

**4. Comprueba.** **1980** filas (2010 − 30 líneas de pedidos descartados).

**Por qué este paso.** Si haces left desde items te quedas en 2010.


### Paso 3 — GMV y mes

**1. Crea una celda Markdown** en *tu* notebook. Explica con tus palabras (puedes partir de esto):

> La fórmula es una columna. Si discount es 1.50, el GMV sale negativo: suciedad que tapas en el siguiente lab.

**2. Crea una celda de código** debajo y escribe:

```python
from pyspark.sql.functions import date_format

lines = (
    lines.withColumn(
        "gmv_line",
        col("qty") * col("unit_price") * (1 - col("discount")),
    ).withColumn("order_month", date_format(col("order_ts"), "yyyy-MM"))
)
lines.select("order_id", "qty", "unit_price", "discount", "gmv_line", "order_month").show(5)
print("gmv nulos", lines.where(col("gmv_line").isNull()).count())
print("gmv < 0", lines.where(col("gmv_line") < 0).count())
```

**3. Ejecuta** esa celda (`Shift+Enter`). Espera a que deje de verse `[*]`.

**4. Comprueba.** `gmv nulos 0` · `gmv < 0` **13**. `order_month` tipo string `2024-01` … `2024-12`.

**Por qué este paso.** No “arregles” el negativo aquí. Quieres verlo.


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

Cero nulos de `gmv_line` en las 1980 filas; 12 meses de 2024; **13** GMV negativos.
Deja `lines` en el notebook: lo usas en M03-02 (o rehaz estos 3 pasos).


## Mejora — Pedido de alto valor

Crea `is_high_value` = `gmv_line >= 500` y cuenta los true (GMV aún sin capar).

<details>
<summary>Si te atascas, mira una solución</summary>

```python
lines.withColumn("is_high_value", col("gmv_line") >= 500).where("is_high_value").count()
# 506
```

</details>


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| 2010 tras el join | Hiciste left | Inner contra orders_clean → 1980 |
| gmv_line string raro | No casteaste unit_price en M02 | Relee el staging |
| order_month nulo | order_ts no parseó | Vuelve a M02-02 (coalesce de dos formatos) |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M03-02 reglas](03-lab-reglas-negocio.ipynb).
